In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from factor_analyzer import FactorAnalyzer, calculate_kmo
import pandas as pd

resmat = pd.read_pickle("../data/resmat.pkl")

# ===================================================================
# Example: Kaiser Rule for choosing number of factors
# Input: resmat (binary response matrix: persons x items)
# ===================================================================

# Step 1: Compute correlation matrix
# (tetrachoric is better for binary data, but here we use Pearson
# for simplicity unless you already computed tetrachoric)
corr = np.corrcoef(~np.isnan(resmat) * resmat, rowvar=False)

# Step 2: Factor analysis to extract eigenvalues
fa = FactorAnalyzer(rotation=None)
# Convert to DataFrame for fillna() method, or handle NaNs differently
resmat_df = pd.DataFrame(resmat)
fa.fit(resmat_df.fillna(0))  # fillna required since FactorAnalyzer doesn't take NaN
ev, v = fa.get_eigenvalues()

# Step 3: Apply Kaiser rule (keep factors with eigenvalue > 1)
kaiser_factors = np.sum(ev > 1)
print(f"Kaiser rule suggests retaining {kaiser_factors} factors.")

# Step 4: Scree plot
plt.figure(figsize=(8,5))
plt.plot(range(1, len(ev)+1), ev, "o-", markersize=6)
plt.axhline(y=1, color="r", linestyle="--", label="Kaiser cutoff (λ=1)")
plt.title("Scree Plot (Eigenvalues)")
plt.xlabel("Factor Number")
plt.ylabel("Eigenvalue")
plt.legend()
plt.show()

In [7]:
import pandas as pd
import numpy as np

# --- 1. Setup: Create a sample DataFrame similar to yours ---
# Let's imagine 10 participants, 2 scenarios, and 5 items per scenario.
# Your actual data is 183 participants, 22 scenarios, and many items.
scenarios = ['Scenario_A', 'Scenario_B']
items = [f'item{i+1}' for i in range(5)]
multi_index = pd.MultiIndex.from_product([scenarios, items], names=['scenario', 'item'])

# Create random binary data
data = np.random.randint(0, 2, size=(10, len(multi_index)))
resmat = pd.DataFrame(data, columns=multi_index)

print("Original Data Structure (a small sample):")
print(resmat.head(3))
print("-" * 50)

# --- 2. Define your parcel mapping (CRUCIAL STEP) ---
# This is based on your theory. Which items group together conceptually?
# For this example, let's say items 1, 3, 5 are one group, and 2, 4 are another.
parcel_mapping = {
    'Parcel_Visual': ['item1', 'item3', 'item5'],
    'Parcel_Logic': ['item2', 'item4']
}


# --- 3. Loop through scenarios and create parcels ---
parceled_results = []
unique_scenarios = resmat.columns.get_level_values('scenario').unique()

for scenario in unique_scenarios:
    print(f"Processing parcels for: {scenario}")
    
    # a. Subset the data for the current scenario
    # The (scenario, slice(None)) syntax selects all columns under a specific scenario.
    scenario_data = resmat.loc[:, (scenario, slice(None))]
    
    # Remove the top level of the MultiIndex to simplify column names (e.g., from ('Scenario_A', 'item1') to 'item1')
    scenario_data.columns = scenario_data.columns.droplevel('scenario')
    
    # Create a new DataFrame to hold the parcels for this scenario
    temp_parceled_df = pd.DataFrame(index=resmat.index)
    
    # b. Apply the parceling logic using the mapping
    for parcel_name, item_list in parcel_mapping.items():
        # Check which items from the mapping exist in the current subset
        existing_items = [item for item in item_list if item in scenario_data.columns]
        if existing_items:
            temp_parceled_df[parcel_name] = scenario_data[existing_items].mean(axis=1)
            
    parceled_results.append(temp_parceled_df)

# --- 4. Combine all results into a final DataFrame ---
# The 'keys' argument will recreate the 'scenario' level in the MultiIndex
final_parceled_data = pd.concat(parceled_results, axis=1, keys=unique_scenarios, names=['scenario', 'parcel'])


print("-" * 50)
print("Final Parceled Data (ready for factor analysis):")
print(final_parceled_data.head())

Original Data Structure (a small sample):
scenario Scenario_A                         Scenario_B                        
item          item1 item2 item3 item4 item5      item1 item2 item3 item4 item5
0                 0     0     1     1     1          1     0     1     0     1
1                 1     1     0     0     1          1     1     1     1     1
2                 0     1     1     1     1          0     1     1     1     1
--------------------------------------------------
Processing parcels for: Scenario_A
Processing parcels for: Scenario_B
--------------------------------------------------
Final Parceled Data (ready for factor analysis):
scenario    Scenario_A                 Scenario_B             
parcel   Parcel_Visual Parcel_Logic Parcel_Visual Parcel_Logic
0             0.666667          0.5      1.000000          0.0
1             0.666667          0.5      1.000000          1.0
2             0.666667          1.0      0.666667          1.0
3             0.333333     